# 5. The genes behind a call

The earlier tutorials asked HECTOR *what* each cell is. This one asks which
genes it used to decide.

`find_marker_genes` measures how much each gene pushed a cell towards the type
it was called, then keeps the genes that push towards one type and not towards
the others. The result is a ranked table of genes per cell type.

**It is not a differential expression test.** That test compares measured
amounts between groups of cells, across every gene you sequenced. This reports
what one trained model relies on, across the 5,000 genes that model reads.
Different question, different answer — so run `scanpy.tl.rank_genes_groups`
as well and join the two on gene identifier.

Nothing extra needs installing.

## Getting the data

We use the multi-platform blood dataset from tutorial 01: the same two donors
profiled on nine sequencing technologies, and — the reason it is the right
dataset here — **cell type labels published by the people who generated it**.

> Ding J, Adiconis X, Simmons SK, *et al.* Systematic comparison of
> single-cell and single-nucleus RNA-sequencing methods.
> *Nature Biotechnology* **38**, 737–746 (2020).
> [doi:10.1038/s41587-020-0465-8](https://doi.org/10.1038/s41587-020-0465-8)

The download is about 70 MB and happens once. If you ran tutorial 01 in this
directory the file is already here.

In [1]:
import pathlib
import shutil
import urllib.request

DATA_URL = ("https://huggingface.co/datasets/polligator/HECTOR/"
            "resolve/main/pbmc_multiplatform.h5ad")
SOURCE = pathlib.Path("pbmc_multiplatform.h5ad")

if not SOURCE.exists():
    request = urllib.request.Request(DATA_URL,
                                     headers={"User-Agent": "hector-tutorial"})
    with urllib.request.urlopen(request) as response, open(SOURCE, "wb") as handle:
        shutil.copyfileobj(response, handle)

## Step 1. Annotate the cells

The same three lines as every tutorial so far. `label_format="name"` gives
readable cell type names rather than ontology identifiers; both work in Step 3.

In [2]:
import anndata as ad
import pandas as pd
import hector

adata = ad.read_h5ad(SOURCE)

predictor = hector.HECTOR("human")
predictions = predictor.predict(adata, label_format="name")
predictor.write_predictions(adata, predictions)

print(f"{adata.obs['hector_prediction'].nunique()} distinct types predicted")
adata.obs["hector_prediction"].value_counts().head(10)

<environment>/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ☑️  MLX encoder loaded
  ☑️  MLX CellTypeEmbedder loaded
  ☑️  MLX HPLHead loaded
  📂 Loaded Memory Bank: 100,000 anchor cells
 🎉 Model loaded successfully! 🥳
   Prediction hardware: GPU (Apple M1 Max, MLX)
   Supported cell types: 1407
  [INFO] Making predictions on 31021 cells...
  Settings: top_k=1, use_grit=True, zero_shot=True
  Step 0: Checking gene IDs...
  Step 1: Gene Matching and Reordering ...
  Gene matching: 4954/5000 (99.1% of reference genes found)
  Step 2: Processing expression data...
  MLX adaptive batch size: 24041 (budget 17.0 GB, floor 5.7 GB)


  Encoding: 100%|██████████████████████████████| 2/2 [00:49<00:00, 24.54s/batch]


  Step 3: Predicting cell types...


  Predicting: 100%|████████████████████████████| 4/4 [00:04<00:00,  1.08s/batch]


  [INFO] Applying GRIT refinement...

  GRIT REFINEMENT
  Metric          Min      5%       25%      Median   75%      95%      Max     
  ---------------------------------------------------------------------------
  Entropy         2.49     2.78     3.03     3.24     3.38     3.55     4.20    
  ---------------------------------------------------------------------------

  [Auto-Thresholding]: Detected Threshold: 3.3238. Refining the lower 35.1% of the prediction distribution

  Step 4: Refining 10,899 cells using all 31,021 cells as neighbors ...


  kNN graph: 100%|██████████████████████████████| 8/8 [00:02<00:00,  3.20tile/s]


  ✅ Predictions complete!
  Step 5: Formatting results...
  ✅ Results formatted as DataFrame with shape (31021, 3)
  Rare-cell rollup summary: threshold=20 cells (min(1.0% of 31021, 20)), 111 type(s) bucketed as 'Rare Cells', 336 cell(s) affected
  Annotated adata.obs with 5 columns (prefix='hector_', 31021/31021 cells matched)
203 distinct types predicted


hector_prediction
classical monocyte                                                            4084
natural killer cell                                                           1882
effector memory CD8-positive, alpha-beta T cell                               1363
CD4-positive, alpha-beta memory T cell, CD45RO-positive                       1306
gamma-delta T cell                                                            1219
effector memory CD8-positive, alpha-beta T cell, terminally differentiated    1066
naive B cell                                                                  1060
naive thymus-derived CD4-positive, alpha-beta T cell                          1046
CD4-positive, alpha-beta cytotoxic T cell                                     1037
platelet                                                                       955
Name: count, dtype: int64

## Step 2. Group the predictions before ranking them

**Do not hand that column straight to `find_marker_genes`.** This is the step
people skip, and it is the difference between a useful table and a worthless
one, so it is worth understanding why.

A gene's rank comes from `spec`: this type's average attribution for the gene,
**minus the strongest average any other type gives it**. Every type competes
with every other. Two consequences follow.

*Overlapping types cancel each other out.* HECTOR answered here with around
two hundred types, and they nest — `T cell`, `mature alpha-beta T cell` and
`effector memory CD8-positive, alpha-beta T cell` are three answers about the
same cells. Whatever gene marks one marks its neighbours too, so the
subtraction wipes it out and all three come back with nothing.

*Small types have nothing to average over.* Two hundred types over thirty
thousand cells leaves a long tail holding a handful of cells each.

Rather than take that on trust, run it. This is the call Step 3 makes, handed
the prediction column directly, printing one type you can judge for yourself.

In [3]:
unfolded = predictor.find_marker_genes(
    adata, label_key="hector_prediction", sample_key="platform", top_n=10,
)
nk = unfolded[unfolded["cell_type_name"] == "natural killer cell"]
print(", ".join(nk.sort_values("rank")["gene_name"]))

  Step 0: Checking gene IDs...
  Step 1: Gene Matching and Reordering ...
  Gene matching: 4954/5000 (99.1% of reference genes found)
  Step 2: Processing expression data...
  [markers] attributions: 31021 cells, 203 types...


  IxG attribution (MLX): 100%|██████████| 122/122 [00:03<00:00, 35.46batch/s]

  [markers] aggregating + confidence (auto)...


  [markers] assembling + containing...
CXCR1, PCDHGA3, KRT23, CCL7, TMEM47, DNLZ, FAM216B, LBX2, TH, KIF20A


Compare that with the same cell type in Step 6, after grouping. The failure is
**silent** — a full table comes back either way, with ranks, p-values and every
column populated.

So fold the predictions into a few groups that do not overlap. The rules below
read only what each label states about itself, and are written out rather than
hidden in a helper file so you can check them, and disagree. Tutorial 01
Step 8 uses the same rules to compare HECTOR against the published labels.

In [4]:
def to_broad_type(label):
    """Fold a HECTOR label into one of nine broad blood cell types.

    Returns None for anything outside them, which those cells are then dropped
    for — see the count printed below.
    """
    text = str(label).lower()
    if "plasmacytoid dendritic" in text:
        return "plasmacytoid dendritic cell"
    if "dendritic" in text:
        return "dendritic cell"
    if "platelet" in text or "megakaryocyte" in text:
        return "megakaryocyte"
    if "monocyte" in text and ("cd16-positive" in text or "non-classical" in text):
        return "non-classical monocyte"
    if "monocyte" in text or "macrophage" in text:
        return "classical monocyte"
    if "natural killer" in text:
        return "natural killer cell"
    if "cd8" in text:
        return "CD8-positive, alpha-beta T cell"
    if "cd4" in text:
        return "CD4-positive, alpha-beta T cell"
    if "b cell" in text or "plasma" in text or "b lineage" in text:
        return "B cell"
    return None


adata.obs["broad_type"] = adata.obs["hector_prediction"].map(to_broad_type)

dropped = adata.obs["broad_type"].isna()
print(f"{adata.obs['broad_type'].nunique()} groups; "
      f"{int(dropped.sum())} cells ({100 * dropped.mean():.1f}%) outside them")
adata.obs["broad_type"].value_counts()

9 groups; 3982 cells (12.8%) outside them


broad_type
CD8-positive, alpha-beta T cell    6218
CD4-positive, alpha-beta T cell    5684
B cell                             4827
classical monocyte                 4741
natural killer cell                2244
dendritic cell                     1282
megakaryocyte                       986
non-classical monocyte              861
plasmacytoid dendritic cell         196
Name: count, dtype: int64

**Every one of these nine names is a cell type the model knows.** That is not
cosmetic. `find_marker_genes` scores your labels inside the model's own
vocabulary, so each must be a Cell Ontology term — a readable name such as
`classical monocyte`, or its identifier `CL:0000860`. A label the model does
not recognise is not an error; those cells are quietly left out of the
calculation, which is worse. The count printed above is how you check.

The cells dropped here are the ones with no equivalent among the nine —
largely the precise T cell populations tutorial 01 Step 8 ran into, such as
gamma-delta and mucosal-associated invariant T cells.

## Step 3. Rank the genes

One call. It runs over every labelled cell, so give it a moment.

`sample_key` is the argument worth pausing on. It names the unit your data
replicates over — normally the donor or the patient — and it changes how the
`confidence` column is computed: with it, each sample contributes one data
point, instead of each cell. That matters because cells from one donor are not
independent observations, and treating them as though they were makes any test
look far more certain than it is.

**Here we pass `platform`, which is deliberately not the biological unit.**
This dataset holds the same two donors run on nine technologies, so treating
each technology as a replicate asks a question you cannot usually ask: *does
this gene still identify this cell type whichever machine produced the data?*
Step 5 reads the answer. On your own data, pass your donor or patient column.

In [5]:
markers = predictor.find_marker_genes(
    adata,
    label_key="broad_type",
    sample_key="platform",
    top_n=50,
)
print(f"{len(markers)} rows, {markers['cell_type'].nunique()} cell types")

  Step 0: Checking gene IDs...
  Step 1: Gene Matching and Reordering ...
  Gene matching: 4954/5000 (99.1% of reference genes found)
  Step 2: Processing expression data...
  [markers] attributions: 27039 cells, 9 types...


  IxG attribution (MLX): 100%|██████████| 106/106 [00:02<00:00, 38.07batch/s]

  [markers] aggregating + confidence (auto)...


  [markers] assembling + containing...
450 rows, 9 cell types


## Step 4. Read one block

The table is one block per cell type, each ranked. Any block would do here; the
columns are the point.

In [6]:
block = markers[markers["cell_type_name"] == "megakaryocyte"]
block[["gene_name", "spec", "contrast_score", "confidence",
       "sample_reproducibility"]].head(10)

,gene_name,spec,contrast_score,confidence,sample_reproducibility
100,PPBP,0.008038,2.336768,0.001823,0.777778
101,SPARC,0.006381,21.371960,0.003813,0.777778
102,GNG11,0.006047,14.000579,0.001790,0.777778
103,ITGA2B,0.004524,35.243074,0.003063,1.000000
104,TUBB1,0.004221,5.640753,0.058563,0.555556
105,SLC25A37,0.004091,100.000000,0.096884,0.333333
106,NXF3,0.003246,27.460986,0.026434,0.666667
107,CD9,0.002306,26.986122,0.037657,0.888889
108,GP9,0.002236,100.000000,0.005048,0.888889
109,TREML1,0.002216,49.475293,0.021181,0.555556


* **`spec`** — specificity, and **the column the ranking is built on**. It is
  this type's average attribution for the gene minus the strongest average any
  *other* type gives it. So a gene the model leans on everywhere scores low
  however important it is, and only genes that separate this type from the rest
  survive. Rows with `spec` at or below zero are dropped.
* **`attribution_score`** — the raw average, signed. Positive means the gene
  pushed cells towards this type, negative means it pushed them away.
* **`contrast_score`** — the same comparison as a ratio rather than a
  difference, capped at 100. Read it as "how many times more this type than the
  next strongest". It is there to be interpretable; it does not affect the
  order.
* **`confidence`** — a p-value for the gene being genuinely elevated in this
  type. Smaller is stronger.
* **`confidence_basis`** — which test produced it. Here it reads
  `across_sample(n=9)` and similar, because `sample_key` was given and each
  type appears in six to nine of the nine platforms.
* **`sample_reproducibility`** — the fraction of samples in which the gene held
  up as a marker. Step 5 is about this column.
* **`gene_id` / `gene_name`** — Ensembl identifier and symbol.

**Whether a block is any good is a judgement you have to make**, and this
method rests on it. Read the top of the list and ask whether the names are the
ones you would expect for that population; if you are not sure, look a few of
them up. That check is not a formality — Step 6 shows a group where it fails.

**One thing the table cannot tell you is what is missing.** PF4, a gene
routinely named alongside PPBP for this lineage, is not here. It is absent
because it is not one of the 5,000 genes this model reads, not because the
model weighed it and dismissed it. Absence from this table is never evidence
about a gene.

## Step 5. Does the gene hold across platforms?

`sample_reproducibility` is the column that stops you over-reading a single
number, and passing `platform` in Step 3 aimed it at technology.

Look back at the megakaryocyte block, at the two genes tied on the highest
value the contrast score can take, 100. **SLC25A37** even ranks above **GP9**,
on a higher `spec`. By the two columns that catch the eye, it is the better
result of the pair.

It held in three of the nine platforms — reproducibility **0.33** — against
GP9's eight, at **0.89**. Change the instrument and one of these two survives.
Nothing else on the row would have warned you.

The pattern holds across the whole table, which is the useful part: the higher
a gene ranks, the better it travels — on average, and not row by row, which is
exactly why the column is worth reading per row.

In [7]:
top_ranked = markers[markers["rank"] <= 10]["sample_reproducibility"]
bottom = markers[markers["rank"] > 40]["sample_reproducibility"]
print(f"mean reproducibility, ranks 1-10 : {top_ranked.mean():.2f}")
print(f"mean reproducibility, ranks 41-50: {bottom.mean():.2f}")
print(f"\nmarkers holding in every platform: "
      f"{100 * (markers['sample_reproducibility'] == 1).mean():.0f}%")
print(f"markers holding in under half     : "
      f"{100 * (markers['sample_reproducibility'] < 0.5).mean():.0f}%")

mean reproducibility, ranks 1-10 : 0.66
mean reproducibility, ranks 41-50: 0.38

markers holding in every platform: 2%
markers holding in under half     : 49%


Two donors would not have shown you any of this. With `sample_key="donor"` on
this file the column can only take three values — 0, 0.5 and 1 — and most
genes land on 1. A reproducibility score is only as informative as the number
of samples behind it, so read it accordingly, and treat it as a warning flag
rather than a measurement when you have two or three samples.

## Step 6. Do the other types make sense?

The real test of a method like this is whether it recovers what is already
known, on types you can check. Here are the top ten for each.

In [8]:
for name, rows in markers.groupby("cell_type_name"):
    genes = rows.sort_values("rank")["gene_name"].head(10).tolist()
    print(f"{name}\n   {', '.join(genes)}\n")

B cell
   CXCR4, TCL1A, BIRC3, CD74, HLA-DRA, VPREB3, LAPTM5, ZFP36L1, CD19, BANK1

CD4-positive, alpha-beta T cell
   IL7R, JUNB, TSC22D3, PIK3IP1, CD40LG, IL6ST, ZFP36L2, ITM2A, CD69, CCND3

CD8-positive, alpha-beta T cell
   CD8A, CD8B, MT-ND4L, ACTB, MYL12A, GZMK, DUSP2, YWHAQ, DAZAP2, PATL2

classical monocyte
   S100A8, VCAN, DUSP1, FOS, KLF6, TYMP, CSF3R, CD14, TNFAIP2, THBS1

dendritic cell
   MT-CO3, TXNIP, VIM, B2M, MT-CYB, MT-ATP6, TMSB4X, FCER1A, RPL7, EIF4B

megakaryocyte
   PPBP, SPARC, GNG11, ITGA2B, TUBB1, SLC25A37, NXF3, CD9, GP9, TREML1

natural killer cell
   PRF1, CD247, NKG7, SPON2, IL2RB, KLRD1, CTSW, HOPX, CST7, CLIC3

non-classical monocyte
   MS4A7, SAT1, COTL1, CFD, PSAP, LRRC25, GPX1, FAM110A, TPM3, FGR

plasmacytoid dendritic cell
   UGCG, IRF7, IRF8, CCDC50, JCHAIN, PLEK, SERPINF1, RNASE6, PTCRA, ALOX5AP



Eight of the nine lists are made of names you can check against what is already
published for these populations — CD19 and BANK1 under **B cell**, NKG7 and
KLRD1 under **natural killer cell**, CD14 and VCAN under **classical
monocyte**, and a different set under **non-classical monocyte**. Do that check
yourself; nothing here does it for you, and none of it was supplied to the
model.

**`dendritic cell` is the one that stands out, and it is worth reading rather
than skipping.** Its top ten are MT-CO3, TXNIP, VIM, B2M, MT-CYB, MT-ATP6,
TMSB4X, FCER1A, RPL7 and EIF4B. Six of those are mitochondrial or ribosomal
genes, which every cell in the file expresses.

Read that as the table reporting that **almost nothing separated this group
from the others** — not as a marker list. When the top of a block is made of
genes every cell expresses, the specificity subtraction found little to keep.
The usual reasons are that the group is too broad, or that it pools populations
with no shared programme; both are decided in Step 2, so that is where to go
and look.

## Step 7. Which markers move together

`find_gene_networks` takes the table you already have, and for each cell type
builds a co-expression graph over that type's top genes and splits it into
modules. Where the marker list is a ranking, this is the structure inside it.

Pass a **focused** table — one cell type, or a few. It iterates over every type
present, so handing it the whole table does far more work than you asked for.

In [9]:
megakaryocyte = markers[markers["cell_type_name"] == "megakaryocyte"]

networks = predictor.find_gene_networks(
    adata,
    megakaryocyte,
    label_key="broad_type",
    top_k=50,
    use_gprofiler=False,
)

for cell_type, network in networks.items():
    name = predictor.id_to_name_map.get(cell_type, cell_type)
    print(f"{name}: {len(network['modules'])} modules")
    for module in network["modules"]:
        print(f"   {', '.join(module['gene_names'])}")

megakaryocyte: 3 modules
   PPBP, GNG11, NXF3, CD9, TIMP1, FKBP1A, NGFRAP1
   TUBB1, PTGS1, RUFY1, TSC22D1, MPP1
   SPARC, ITGA2B, GP9, TREML1


A module is a set of genes whose counts rise and fall together across the cells
of that type. That is all it is: **not a pathway, and not a claim about
mechanism.** Step 4's ranked list interleaves whatever structure exists; this
pulls it apart.

What to do with it is look at whether a module corresponds to something you
already recognise, and treat the ones that do as worth following up. The
enrichment step below is what turns that impression into a named pathway.

`use_gprofiler=False` turns off the last step, which would name a pathway for
each module. That step contacts g:Profiler, an online service, so it is off
here to keep the tutorial from depending on a website being up. Leave it at its
default of `True` to have your modules annotated.

## What this table is and is not

* **It covers 5,000 genes, not your whole file.** That is the set the model
  reads. A gene absent from it cannot appear here however important it is in
  your biology, and its absence says nothing about it.
* **It describes the model, not the tissue.** These are the genes HECTOR relies
  on to recognise a type. A gene that identifies the population perfectly but
  that the model happens not to use will not be listed.
* **Your grouping decides the answer.** `spec` compares each type against the
  other labels you supplied, so a different grouping gives a different table.
  Step 2 is not tidying-up before the real work; it is part of the question.
* **`confidence` is a p-value about a model's behaviour**, not evidence for a
  biological claim. Rank by `spec`, sanity-check with
  `sample_reproducibility`, and treat the whole table as a hypothesis
  generator.
* **Save it.** The table is the durable output; regenerating it needs the model
  and the data together.

In [10]:
markers.to_csv("05_marker_genes.csv", index=False)

## Where next

That is the last of the tutorials. `docs/installation.md` covers setting HECTOR
up on a new machine, and every function used across these five has a docstring
with the arguments not touched on here — `help(predictor.find_marker_genes)` in
a notebook cell is the quickest way in.